# Draft-model mistake analysis

Analyzes **which tokens the EAGLE3 draft gets wrong** on eval benchmarks, to decide *what training corpus would help*.

Input is the `mistakes.jsonl` from `score_mistakes.py` (run on the GPU box). This notebook needs no GPU. Three lenses:

1. **Token categories** — what *kind* of token the draft misses (code punctuation, numbers, LaTeX/math, identifiers, prose …), contrasting benchmarks.
2. **Rare-in-training cross-reference** — missed tokens that were rare/absent in the training data (`token_freq.pt`). This is the direct corpus signal.
3. **Position & TTT-depth** — where in the sequence / at which draft depth it fails.

In [ ]:
# --- config: edit these paths ---
MISTAKES_JSONL = "out/run1_mistakes.jsonl"                    # from score_mistakes.py
VERIFIER       = "/sms-scratch/checkpoints/gemma-4-31B-it"    # for token decoding
TOKEN_FREQ_PT  = "/sms-scratch/ravira/datasets/kimi-mtp-dataset/token_freq.pt"

import matplotlib.pyplot as plt
import pandas as pd
import mistake_lib as ML
pd.set_option("display.max_rows", 60)

In [ ]:
# --- load + enrich ---
from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained(VERIFIER, trust_remote_code=True)
freq = ML.load_token_freq(TOKEN_FREQ_PT)

df = ML.load_mistakes(MISTAKES_JSONL)
df = ML.attach_token_strings(df, tok)
df = ML.add_categories(df)
df = ML.add_training_freq(df, freq)
print(len(df), "token records across", df.benchmark.nunique(), "benchmarks")
df.head()

## Headline acceptance
`step0_accept` is the base accept rate (dominates end-to-end acceptance length).

In [ ]:
ML.summary(df)

## Lens 1 — token categories
Where do step-0 mistakes concentrate, and how does accuracy per category differ across benchmarks? High `mistake_share` = that category is where errors pile up.

In [ ]:
cat = ML.token_category_breakdown(df, ttt_step=0)
cat

In [ ]:
piv = cat.pivot(index="category", columns="benchmark", values="accuracy")
ax = piv.plot(kind="bar", figsize=(11,4))
ax.set_ylabel("step-0 accuracy"); ax.set_title("Draft accuracy by token category")
ax.set_ylim(0,1); plt.xticks(rotation=30, ha="right"); plt.tight_layout(); plt.show()

In [ ]:
piv2 = cat.pivot(index="category", columns="benchmark", values="mistake_share").fillna(0)
ax = piv2.plot(kind="bar", figsize=(11,4))
ax.set_ylabel("share of mistakes"); ax.set_title("Mistake concentration by category")
plt.xticks(rotation=30, ha="right"); plt.tight_layout(); plt.show()

## Lens 2 — rare-in-training cross-reference
**The corpus signal.** If accuracy collapses on `absent`/`rare` tokens, the draft is failing on content it barely saw in training — add a corpus rich in those.

In [ ]:
fb = ML.accuracy_by_freq_bucket(df, ttt_step=0)
piv = fb.pivot(index="freq_bucket", columns="benchmark", values="accuracy")
ax = piv.plot(kind="bar", figsize=(9,4))
ax.set_ylabel("step-0 accuracy"); ax.set_ylim(0,1)
ax.set_title("Accuracy vs. training-set frequency of the target token")
plt.xticks(rotation=0); plt.tight_layout(); plt.show()
fb

### Shortlist: frequently-missed AND rare-in-training tokens
These are the concrete tokens to go find more training data for.

In [ ]:
ML.top_missed_rare_tokens(df, ttt_step=0, max_train_freq=100, top_n=40)

## Lens 3 — TTT depth & position

In [ ]:
dp = ML.depth_profile(df)
fig, ax = plt.subplots(1,2, figsize=(13,4))
for b, g in dp.groupby("benchmark"):
    ax[0].plot(g.ttt_step, g.full_acc, marker="o", label=b)
    ax[1].plot(g.ttt_step, g.cond_acc, marker="o", label=b)
ax[0].set_title("full acc by TTT step"); ax[1].set_title("conditional (chained) acc by TTT step")
for a in ax: a.set_xlabel("ttt_step"); a.set_ylim(0,1); a.legend()
plt.tight_layout(); plt.show()
dp

In [ ]:
pp = ML.position_profile(df, ttt_step=0, bins=10)
piv = pp.pivot(index="pos_bin", columns="benchmark", values="accuracy")
ax = piv.plot(marker="o", figsize=(10,4))
ax.set_ylabel("step-0 accuracy"); ax.set_ylim(0,1)
ax.set_title("Accuracy across normalized position (degradation deep in generation?)")
plt.tight_layout(); plt.show()

---
### Reading this for corpus selection
- **Lens 2** is the money shot: a big accuracy gap between `common` and `absent`/`rare` on gpqa/livecodebench means those benchmarks lean on tokens under-represented in `kimi-mtp`. The shortlist names them.
- **Lens 1** tells you the *theme* (code punctuation vs. LaTeX vs. prose) so you can pick a corpus (more code, more math/science) rather than chase individual tokens.
- **Lens 3** separates "draft is weak everywhere" (low step-0) from "draft decays with depth/length" (fine at step-0, collapses deeper) — different fixes.